In [7]:
from tkinter import *
from tkinter import filedialog, messagebox
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier, PassiveAggressiveClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import seaborn as sns

# ================= GLOBALS =================
raw_dataset = None
dataset = None
X = y = None
x_train = x_test = y_train = y_test = None
mlmodel = None
target_name = None
le = None
scaler = None

accuracy, precision, recall, fscore = [], [], [], []
algorithm_names = []

graph_canvas = None

# ================= FUNCTIONS =================
def clear_output():
    global graph_canvas
    output_text.delete('1.0', END)
    if graph_canvas:
        graph_canvas.get_tk_widget().destroy()
        graph_canvas = None

def uploadDataset():
    global raw_dataset, dataset
    file = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
    if file:
        clear_output()
        raw_dataset = pd.read_csv(file)
        dataset = raw_dataset.copy()
        output_text.insert(END, "Dataset Loaded Successfully\n\n")
        output_text.insert(END, str(dataset.head()) + "\n\n")

def Preprocess_Dataset():
    global dataset, raw_dataset, X, y, target_name, le
    if raw_dataset is None:
        messagebox.showwarning("Warning", "Upload dataset first")
        return

    dataset = raw_dataset.copy()
    dataset.columns = dataset.columns.str.strip()

    for col in dataset.columns:
        if col.lower() in ['prakriti', 'class', 'target', 'label']:
            target_name = col
            break

    if target_name is None:
        output_text.insert(END, "Target column not found!\n")
        return

    dataset.dropna(inplace=True)

    le = LabelEncoder()
    for col in dataset.columns:
        if col != target_name and dataset[col].dtype == 'object':
            dataset[col] = le.fit_transform(dataset[col])

    dataset[target_name] = le.fit_transform(dataset[target_name])

    X = dataset.drop(target_name, axis=1)
    y = dataset[target_name]

    clear_output()
    output_text.insert(END, "Preprocessing Completed Successfully\n\n")

def Train_Test_Splitting():
    global X, y, x_train, x_test, y_train, y_test, scaler
    if X is None:
        messagebox.showwarning("Warning", "Run preprocessing first")
        return

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    x_train, x_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )

    clear_output()
    output_text.insert(END, f"Training Samples : {len(x_train)}\n")
    output_text.insert(END, f"Testing Samples  : {len(x_test)}\n\n")

def Calculate_Metrics(name, pred):
    a = accuracy_score(y_test, pred) * 100
    p = precision_score(y_test, pred, average='weighted') * 100
    r = recall_score(y_test, pred, average='weighted') * 100
    f = f1_score(y_test, pred, average='weighted') * 100

    accuracy.append(a)
    precision.append(p)
    recall.append(r)
    fscore.append(f)
    algorithm_names.append(name)

    output_text.insert(END, f"{name}\n")
    output_text.insert(END, f"Accuracy  : {a:.2f}%\n")
    output_text.insert(END, f"Precision : {p:.2f}%\n")
    output_text.insert(END, f"Recall    : {r:.2f}%\n")
    output_text.insert(END, f"F1 Score  : {f:.2f}%\n\n")

def plot_confusion_matrix(name, pred):
    global graph_canvas
    # Do not clear output here to keep text visible

    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, pred)
    fig = Figure(figsize=(5, 5), dpi=100)
    ax = fig.add_subplot(111)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix for {name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

    graph_canvas = FigureCanvasTkAgg(fig, master=output_frame)
    graph_canvas.draw()
    graph_canvas.get_tk_widget().place(relx=0.5, rely=0.5, anchor=CENTER)  # Place in center, may overlap text

def run_model(model, name):
    global mlmodel
    mlmodel = model
    mlmodel.fit(x_train, y_train)
    pred = mlmodel.predict(x_test)
    clear_output()  # Clear at start
    Calculate_Metrics(name, pred)
    plot_confusion_matrix(name, pred)  # Add plot after text

def sgd_classifier():
    run_model(SGDClassifier(max_iter=1000), "SGD Classifier")

def qda_classifier():
    run_model(QuadraticDiscriminantAnalysis(), "QDA Classifier")

def passive_aggressive():
    run_model(PassiveAggressiveClassifier(), "Passive Aggressive")

def mlp_classifier():
    run_model(MLPClassifier(max_iter=1000), "MLP Classifier")

def predict():
    global mlmodel, X, scaler, le
    if mlmodel is None:
        messagebox.showwarning("Warning", "Train a model first")
        return
    if X is None or scaler is None or le is None:
        messagebox.showwarning("Warning", "Run preprocessing and train-test split first")
        return

    features = list(X.columns)
    pred_window = Toplevel(main)
    pred_window.title("Input Features for Prediction")
    pred_window.geometry("400x600")

    entries = {}
    for i, feat in enumerate(features):
        Label(pred_window, text=feat).grid(row=i, column=0, padx=10, pady=5)
        entry = Entry(pred_window)
        entry.grid(row=i, column=1, padx=10, pady=5)
        entries[feat] = entry

    def do_predict():
        try:
            input_data = [float(entries[feat].get()) for feat in features]
            scaled_input = scaler.transform([input_data])
            pred = mlmodel.predict(scaled_input)
            predicted_class = le.inverse_transform(pred)[0]
            messagebox.showinfo("Prediction", f"Predicted Prakriti: {predicted_class}")
        except ValueError:
            messagebox.showerror("Error", "Invalid input. Please enter numerical values.")

    Button(pred_window, text="Predict", command=do_predict).grid(row=len(features), column=0, columnspan=2, pady=10)

def graph():
    global graph_canvas

    if len(accuracy) < 2:
        messagebox.showwarning("Warning", "Run at least two classifiers")
        return

    clear_output()

    fig = Figure(figsize=(4,3), dpi=100)
    ax = fig.add_subplot(111)

    metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]

    for i, name in enumerate(algorithm_names):
        scores = [accuracy[i], precision[i], recall[i], fscore[i]]
        ax.plot(metrics, scores, marker='o', label=name)

    ax.set_ylabel("Score (%)")
    ax.set_title("Classifier Performance Comparison")
    ax.legend()

    graph_canvas = FigureCanvasTkAgg(fig, master=output_frame)
    graph_canvas.draw()
    graph_canvas.get_tk_widget().place(relx=0.5, rely=0.5, anchor=CENTER)

# ================= GUI =================
main = Tk()
main.title("Ayurvedic Prakriti Classification System")
main.attributes('-fullscreen', True)  # Make window full screen

# Get screen size for image resizing
screen_width = main.winfo_screenwidth()
screen_height = main.winfo_screenheight()

# ---------- FULL BACKGROUND IMAGE ----------
try:
    from PIL import Image, ImageTk
    img = Image.open(r"C:\Users\LAHARI T\OneDrive\Attachments\Desktop\project\prakriti.jpg.jpeg")
    img = img.resize((screen_width, screen_height), Image.LANCZOS)  # Resize to full screen size
    bg_image = ImageTk.PhotoImage(img)
    bg_label = Label(main, image=bg_image)
    bg_label.place(x=0, y=0, relwidth=1, relheight=1)
except:
    main.configure(bg="#E0F2F1")

# ---------- TITLE ----------
Label(
    main,
    text="Ayurvedic Prakriti Classification System",
    bg="#00897B",
    fg="white",
    font=('Times New Roman', 18, 'bold'),
    height=2,
    width=120
).place(x=0, y=5)

# ---------- OUTPUT PANEL ----------
output_frame = Frame(main, bg="#E0F2F1", bd=2, relief=RIDGE)
output_frame.place(x=150, y=100, width=1000, height=500)

output_text = Text(output_frame, font=('Times New Roman', 12))
output_text.pack(fill=BOTH, expand=True)

# ---------- BUTTON STYLE ----------
btn_font = ("Times New Roman", 12, "bold")
btn_width = 18
btn_height = 2

# ---------- ROW 1 ----------
Button(main, text="Dataset", command=uploadDataset,
       width=btn_width, height=btn_height, font=btn_font).place(x=250, y=640)

Button(main, text="Preprocessing", command=Preprocess_Dataset,
       width=btn_width, height=btn_height, font=btn_font).place(x=430, y=640)

Button(main, text="Train Test Split", command=Train_Test_Splitting,
       width=btn_width, height=btn_height, font=btn_font).place(x=610, y=640)

Button(main, text="SGD Classifier", command=sgd_classifier,
       width=btn_width, height=btn_height, font=btn_font).place(x=790, y=640)

Button(main, text="QDA Classifier", command=qda_classifier,
       width=btn_width, height=btn_height, font=btn_font).place(x=970, y=640)

# ---------- ROW 2 ----------
Button(main, text="Passive Aggressive", command=passive_aggressive,
       width=btn_width, height=btn_height, font=btn_font).place(x=250, y=700)

Button(main, text="MLP Classifier", command=mlp_classifier,
       width=btn_width, height=btn_height, font=btn_font).place(x=430, y=700)

Button(main, text="Predict", command=predict,
       width=btn_width, height=btn_height, font=btn_font).place(x=610, y=700)

Button(main, text="Comparison Graph", command=graph,
       width=btn_width, height=btn_height, font=btn_font).place(x=790, y=700)

Button(main, text="Exit", command=main.destroy,
       width=btn_width, height=btn_height, font=btn_font).place(x=970, y=700)

main.mainloop()